# Experiment: Origen y adquisición del corpus Doppler

**Pregunta.** ¿De qué fuentes sale el audio, con qué licencias, y cuál es el corpus de *entrenamiento* frente al piloto y a los tests de otro dominio?

**Criterio de éxito.** Inventario reproducible (local y Kaggle), tabla tarea↔dataset, y CSV de archivos en `reports/tables/`.


In [14]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

SEED = 7

# Local: data/raw/<corpus>
# Kaggle (Add Input clásico): /kaggle/input/<slug>
# Kaggle (datasets/user): /kaggle/input/datasets/<user>/<slug>/<slug>
IS_KAGGLE = Path("/kaggle/input").exists()

if IS_KAGGLE:
    DATA_ROOT = Path("/kaggle/input")
    FIGURES_DIR = Path("/kaggle/working/reports/figures")
    TABLES_DIR = Path("/kaggle/working/reports/tables")
else:
    here = Path.cwd().resolve()
    REPO_ROOT = here if (here / "data" / "raw").exists() else here.parent
    DATA_ROOT = REPO_ROOT / "data" / "raw"
    FIGURES_DIR = REPO_ROOT / "reports" / "figures"
    TABLES_DIR = REPO_ROOT / "reports" / "tables"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

CORPUS_SLUGS = {
    "sirennet": ("sirennet",),
    "lssiren": ("lssiren",),
    "urbansound8k": ("urbansound8k",),
    "audioset_ev": ("audioset_ev", "audioset-ev"),
}


def is_kaggle() -> bool:
    return IS_KAGGLE


def figures_dir() -> Path:
    return FIGURES_DIR


def tables_dir() -> Path:
    return TABLES_DIR


def resolve_corpus(name: str) -> Path | None:
    candidates: list[Path] = []
    for slug in CORPUS_SLUGS[name]:
        candidates.append(DATA_ROOT / slug)
        datasets = DATA_ROOT / "datasets"
        if datasets.exists():
            for user_dir in datasets.iterdir():
                if not user_dir.is_dir():
                    continue
                candidates.append(user_dir / slug)
                candidates.append(user_dir / slug / slug)
    existing = [path for path in candidates if path.exists()]
    if not existing:
        return None
    return max(existing, key=lambda path: len(path.parts))


@dataclass(frozen=True)
class CorpusPaths:
    sirennet: Path | None
    lssiren: Path | None
    urbansound8k: Path | None
    audioset_ev: Path | None

    def available(self) -> dict[str, Path]:
        found = {
            "sirennet": self.sirennet,
            "lssiren": self.lssiren,
            "urbansound8k": self.urbansound8k,
            "audioset_ev": self.audioset_ev,
        }
        return {key: path for key, path in found.items() if path is not None}


def corpus_paths() -> CorpusPaths:
    return CorpusPaths(
        sirennet=resolve_corpus("sirennet"),
        lssiren=resolve_corpus("lssiren"),
        urbansound8k=resolve_corpus("urbansound8k"),
        audioset_ev=resolve_corpus("audioset_ev"),
    )


print("kaggle:", IS_KAGGLE)
print("DATA_ROOT:", DATA_ROOT)
print("FIGURES_DIR:", FIGURES_DIR)
print("TABLES_DIR:", TABLES_DIR)
print("available:", list(corpus_paths().available()))
SEED


kaggle: False
DATA_ROOT: /home/jeancdevx/dev/doppler/doppler-ml/data/raw
FIGURES_DIR: /home/jeancdevx/dev/doppler/doppler-ml/reports/figures
TABLES_DIR: /home/jeancdevx/dev/doppler/doppler-ml/reports/tables
available: ['sirennet', 'lssiren', 'urbansound8k', 'audioset_ev']


7

In [15]:
# Inventario de archivos
"""Inventario de archivos de audio del corpus Doppler."""

from __future__ import annotations

from pathlib import Path

import pandas as pd

SIRENNET_CLASS_MAP = {
    "ambulance": "ambulance",
    "police": "police",
    "firetruck": "firetruck",
    "fire_truck": "firetruck",
    "fire": "firetruck",
    "traffic": "traffic",
}

LSSIREN_POSITIVE_HINTS = ("emergency", "siren", "ambulance")
LSSIREN_NEGATIVE_HINTS = ("road", "noise", "traffic")
AUDIO_SUFFIXES = {".wav", ".mp3", ".flac", ".ogg", ".m4a"}


def is_audio(path: Path) -> bool:
    return path.suffix.lower() in AUDIO_SUFFIXES


def infer_sirennet_label(path: Path) -> str | None:
    parts = [p.lower() for p in path.parts]
    stem = path.stem.lower()
    for key, label in SIRENNET_CLASS_MAP.items():
        if key in parts or stem.startswith(key) or f"_{key}_" in f"_{stem}_":
            return label
    return None


def infer_lssiren_label(path: Path) -> str | None:
    blob = " ".join(p.lower() for p in path.parts)
    if any(h in blob for h in LSSIREN_POSITIVE_HINTS) and "road" not in blob:
        return "siren"
    if any(h in blob for h in LSSIREN_NEGATIVE_HINTS):
        return "road_noise"
    return None


def scan_sirennet(root: Path) -> pd.DataFrame:
    rows = []
    for path in root.rglob("*"):
        if not path.is_file() or not is_audio(path):
            continue
        rows.append(
            {
                "corpus": "sirennet",
                "path": str(path),
                "relpath": str(path.relative_to(root)),
                "label": infer_sirennet_label(path) or "unknown",
                "task": "multiclass",
            }
        )
    return pd.DataFrame(rows)


LSSIREN_CSV_COLUMNS = [
    "filename",
    "chroma_stft",
    "rmse",
    "spectral_centroid",
    "spectral_bandwidth",
    "rolloff",
    "zero_crossing_rate",
    *[f"mfcc{i}" for i in range(1, 21)],
    "label",
]


def read_lssiren_features(path: Path) -> pd.DataFrame:
    raw = pd.read_csv(path)
    if "filename" in raw.columns:
        return raw
    return pd.read_csv(path, header=None, names=LSSIREN_CSV_COLUMNS)


def scan_lssiren(root: Path) -> pd.DataFrame:
    rows = []
    for path in root.rglob("*"):
        if not path.is_file() or not is_audio(path):
            continue
        label = infer_lssiren_label(path) or "unknown"
        rows.append(
            {
                "corpus": "lssiren",
                "path": str(path),
                "relpath": str(path.relative_to(root)),
                "label": label,
                "task": "binary",
            }
        )
    if rows:
        return pd.DataFrame(rows)

    for csv_path in root.glob("*.csv"):
        feat = read_lssiren_features(csv_path)
        label_col = "label" if "label" in feat.columns else feat.columns[-1]
        name_col = "filename" if "filename" in feat.columns else feat.columns[0]
        for _, row in feat.iterrows():
            raw_label = str(row[label_col]).strip().lower()
            label = "siren" if raw_label in {"ambulance", "siren", "emergency"} else "road_noise"
            rows.append(
                {
                    "corpus": "lssiren",
                    "path": str(csv_path.parent / str(row[name_col])),
                    "relpath": str(row[name_col]),
                    "label": label,
                    "task": "binary",
                    "source": "feature_csv",
                }
            )
    return pd.DataFrame(rows)


def scan_urbansound8k(root: Path) -> pd.DataFrame:
    csv_candidates = list(root.rglob("UrbanSound8K.csv"))
    if csv_candidates:
        meta = pd.read_csv(csv_candidates[0])
        if "class" not in meta.columns and "class_name" in meta.columns:
            meta = meta.rename(columns={"class_name": "class"})
        audio_root = csv_candidates[0].parent.parent / "audio"
        if not audio_root.exists():
            audio_root = root / "audio"
            if not audio_root.exists():
                audio_root = root
        rows = []
        for _, row in meta.iterrows():
            fold = int(row["fold"])
            fname = row["slice_file_name"]
            path = audio_root / f"fold{fold}" / fname
            rows.append(
                {
                    "corpus": "urbansound8k",
                    "path": str(path),
                    "relpath": f"fold{fold}/{fname}",
                    "label": row["class"],
                    "task": "urban_scene",
                    "fold": fold,
                    "fsID": row.get("fsID"),
                    "classID": row.get("classID"),
                    "salience": row.get("salience"),
                }
            )
        return pd.DataFrame(rows)

    rows = []
    for path in root.rglob("*"):
        if not path.is_file() or not is_audio(path):
            continue
        rows.append(
            {
                "corpus": "urbansound8k",
                "path": str(path),
                "relpath": str(path.relative_to(root)),
                "label": path.parent.name,
                "task": "urban_scene",
            }
        )
    return pd.DataFrame(rows)


AUDIOSET_EV_MIDS = {
    "/m/04qvtq": "police",
    "/m/012n7d": "ambulance",
    "/m/012ndj": "firetruck",
}
AUDIOSET_EV_GENERIC_MIDS = {
    "/m/03j1ly",  # Emergency vehicle
    "/m/03kmc9",  # Siren
}


def parse_audioset_mids(raw) -> list[str]:
    if raw is None:
        return []
    try:
        if pd.isna(raw):
            return []
    except (TypeError, ValueError):
        pass
    text = str(raw).strip()
    if not text or text.lower() in {"nan", "none"}:
        return []
    found = []
    for token in (
        text.replace("[", " ")
        .replace("]", " ")
        .replace("'", " ")
        .replace('"', " ")
        .replace(",", " ")
        .split()
    ):
        if token.startswith("/m/") or token.startswith("/g/"):
            found.append(token)
    return found


def map_audioset_labels(mids: list[str]) -> tuple[str, bool, str]:
    mapped: list[str] = []
    for mid in mids:
        label = AUDIOSET_EV_MIDS.get(mid)
        if label and label not in mapped:
            mapped.append(label)
    multi = len(mapped) > 1
    if not mapped:
        if any(mid in AUDIOSET_EV_GENERIC_MIDS for mid in mids):
            type_label = "siren_untyped"
        else:
            type_label = "unknown"
    elif multi:
        type_label = "multi"
    else:
        type_label = mapped[0]
    return type_label, multi, "|".join(mapped)


def _yt_id_from_stem(stem: str) -> str:
    if stem.startswith("Y") and len(stem) > 1:
        return stem[1:]
    return stem


def _audioset_segment_from_path(path: Path) -> str | None:
    parts = [p.lower() for p in path.parts]
    for key in ("unbalanced", "balanced_train", "eval"):
        if key in parts:
            return key
    return None


def _audioset_polarity_from_path(path: Path) -> str | None:
    blob = "/".join(p.lower() for p in path.parts)
    if "negative_files" in blob or "/negatives/" in blob:
        return "negative"
    if "positive_files" in blob or "/positives/" in blob:
        return "positive"
    return None


def _truthy_downloaded(val) -> bool:
    if val is True:
        return True
    if val is False or val is None:
        return False
    try:
        if pd.isna(val):
            return False
    except (TypeError, ValueError):
        pass
    return str(val).strip().lower() in {"true", "1", "yes"}


def _read_audioset_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    df.columns = [str(c).strip() for c in df.columns]
    return df


def scan_audioset_ev(root: Path) -> pd.DataFrame:
    pos_csv = None
    neg_csv = None
    for csv_path in root.rglob("*.csv"):
        name = csv_path.name.lower()
        if name == "ev_positives.csv":
            pos_csv = csv_path
        elif name == "ev_negatives.csv":
            neg_csv = csv_path

    meta_rows = []
    if pos_csv is not None:
        pos = _read_audioset_csv(pos_csv)
        if "downloaded" in pos.columns:
            pos = pos[pos["downloaded"].map(_truthy_downloaded)]
        for _, row in pos.iterrows():
            yt_id = str(row.get("yt_id", row.iloc[0])).strip()
            mids = parse_audioset_mids(row.get("positive_labels", row.get("labels")))
            type_label, multi, joined = map_audioset_labels(mids)
            meta_rows.append(
                {
                    "yt_id": yt_id,
                    "polarity": "positive",
                    "segment_type": str(row.get("segment_type", "")).strip() or None,
                    "downloaded_flag": row.get("downloaded"),
                    "type_label": type_label,
                    "multi_positive": multi,
                    "ev_labels": joined,
                    "mids": "|".join(mids),
                }
            )
    if neg_csv is not None:
        neg = _read_audioset_csv(neg_csv)
        if "downloaded" in neg.columns:
            neg = neg[neg["downloaded"].map(_truthy_downloaded)]
        for _, row in neg.iterrows():
            yt_id = str(row.get("yt_id", row.iloc[0])).strip()
            meta_rows.append(
                {
                    "yt_id": yt_id,
                    "polarity": "negative",
                    "segment_type": str(row.get("segment_type", "")).strip() or None,
                    "downloaded_flag": row.get("downloaded"),
                    "type_label": "urban_negative",
                    "multi_positive": False,
                    "ev_labels": "",
                    "mids": "",
                }
            )
    meta = pd.DataFrame(meta_rows)
    meta_by_id = {}
    if not meta.empty:
        meta_by_id = {str(r["yt_id"]): r for r in meta.to_dict(orient="records")}

    wav_rows = []
    for path in root.rglob("*"):
        if not path.is_file() or not is_audio(path):
            continue
        yt_id = _yt_id_from_stem(path.stem)
        info = meta_by_id.get(yt_id, {})
        polarity = info.get("polarity") or _audioset_polarity_from_path(path) or "unknown"
        segment = info.get("segment_type") or _audioset_segment_from_path(path) or "unknown"
        if polarity == "negative":
            type_label = "urban_negative"
            multi = False
            ev_labels = ""
            task = "binary"
        else:
            type_label = info.get("type_label") or "unknown"
            multi = bool(info.get("multi_positive", False))
            ev_labels = info.get("ev_labels") or ""
            task = "multiclass"
        wav_rows.append(
            {
                "corpus": "audioset_ev",
                "path": str(path),
                "relpath": str(path.relative_to(root)),
                "label": type_label,
                "task": task,
                "yt_id": yt_id,
                "group_id": yt_id,
                "polarity": polarity,
                "segment_type": segment,
                "multi_positive": multi,
                "ev_labels": ev_labels,
                "has_wav": True,
            }
        )

    wav_df = pd.DataFrame(wav_rows)
    if meta.empty:
        return wav_df

    seen = set(wav_df["yt_id"].astype(str)) if not wav_df.empty else set()
    missing_rows = []
    for rec in meta.to_dict(orient="records"):
        if str(rec["yt_id"]) in seen:
            continue
        polarity = rec["polarity"]
        missing_rows.append(
            {
                "corpus": "audioset_ev",
                "path": "",
                "relpath": "",
                "label": rec["type_label"],
                "task": "binary" if polarity == "negative" else "multiclass",
                "yt_id": rec["yt_id"],
                "group_id": rec["yt_id"],
                "polarity": polarity,
                "segment_type": rec["segment_type"] or "unknown",
                "multi_positive": rec["multi_positive"],
                "ev_labels": rec["ev_labels"],
                "has_wav": False,
            }
        )
    if missing_rows:
        wav_df = pd.concat([wav_df, pd.DataFrame(missing_rows)], ignore_index=True)
    return wav_df


def assign_audioset_protocol_split(polarity: str, segment_type: str) -> str:
    pol = (polarity or "").strip().lower()
    seg = (segment_type or "").strip().lower()
    if seg == "eval":
        return "test"
    if pol == "positive" and seg in {"unbalanced", "unbalanced_train"}:
        return "train_pool"
    if pol == "negative" and seg == "balanced_train":
        return "train_pool"
    if pol == "positive" and seg == "balanced_train":
        return "balanced_train_ref"
    return "unused"


## Plan

- Hipótesis: ningún dataset único cubre tipo + calle + distractores con nombre; el train de escala es AudioSet-EV.
- Barrido: AudioSet-EV v2 (train), sireNNet (piloto / test limpio), LSSiren y UrbanSound8K (test de dominio, más adelante).
- Métricas: n_filas, clases, WAV presentes vs solo CSV, licencia.


In [16]:
import pandas as pd

FIG = figures_dir()
TAB = tables_dir()

catalog = pd.DataFrame([
    {
        "corpus": "AudioSet-EV v2",
        "role": "train_escala",
        "task": "binary+multiclass",
        "n_nominal": 28816,
        "classes": "police / ambulance / firetruck vs urban negatives (multi-etiqueta)",
        "license": "subset AudioSet (YouTube); uso de investigacion",
        "source": "https://doi.org/10.5281/zenodo.18668076",
        "citation": "Giacomelli & Rinaldi, 2025, 10.5281/zenodo.18668076",
        "notes": "Clips ~10 s, 32 kHz, mono. Splits oficiales balanced_train / eval / unbalanced.",
    },
    {
        "corpus": "sireNNet",
        "role": "piloto_test_limpio",
        "task": "binary+multiclass",
        "n_nominal": 1675,
        "classes": "ambulance, police, firetruck, traffic",
        "license": "CC BY 4.0",
        "source": "https://data.mendeley.com/datasets/j4ydzzv4kb/1",
        "citation": "Shah & Singh, 2023, 10.17632/j4ydzzv4kb.1",
        "notes": "No entra al fit. Release ya aumentado; 3 s / 44.1 kHz estéreo.",
    },
    {
        "corpus": "LSSiren",
        "role": "test_dominio_calle",
        "task": "binary",
        "n_nominal": 1800,
        "classes": "siren, road_noise",
        "license": "CC BY 4.0",
        "source": "https://doi.org/10.6084/m9.figshare.19291472",
        "citation": "Asif et al., Sci Data 2022, 10.1038/s41597-022-01727-2",
        "notes": "No se descarga WAV en esta fase. Karachi; sesgo a ambulancia.",
    },
    {
        "corpus": "UrbanSound8K",
        "role": "test_distractores",
        "task": "urban_scene / binary_generic_siren",
        "n_nominal": 8732,
        "classes": "10 clases urbanas incl. siren (~929)",
        "license": "CC BY-NC 4.0",
        "source": "https://zenodo.org/records/1203745",
        "citation": "Salamon, Jacoby & Bello, 2014, 10.5281/zenodo.1203745",
        "notes": "WAV no en esta fase. Usar folds oficiales y fsID.",
    },
])
catalog.to_csv(TAB / "corpus_catalog.csv", index=False)
catalog


,corpus,role,task,n_nominal,classes,license,source,citation,notes
0,AudioSet-EV v2,train_escala,binary+multiclass,28816,police / ambulance / firetruck vs urban negati...,subset AudioSet (YouTube); uso de investigacion,https://doi.org/10.5281/zenodo.18668076,"Giacomelli & Rinaldi, 2025, 10.5281/zenodo.186...","Clips ~10 s, 32 kHz, mono. Splits oficiales ba..."
1,sireNNet,piloto_test_limpio,binary+multiclass,1675,"ambulance, police, firetruck, traffic",CC BY 4.0,https://data.mendeley.com/datasets/j4ydzzv4kb/1,"Shah & Singh, 2023, 10.17632/j4ydzzv4kb.1",No entra al fit. Release ya aumentado; 3 s / 4...
2,LSSiren,test_dominio_calle,binary,1800,"siren, road_noise",CC BY 4.0,https://doi.org/10.6084/m9.figshare.19291472,"Asif et al., Sci Data 2022, 10.1038/s41597-022...",No se descarga WAV en esta fase. Karachi; sesg...
3,UrbanSound8K,test_distractores,urban_scene / binary_generic_siren,8732,10 clases urbanas incl. siren (~929),CC BY-NC 4.0,https://zenodo.org/records/1203745,"Salamon, Jacoby & Bello, 2014, 10.5281/zenodo....",WAV no en esta fase. Usar folds oficiales y fsID.


## Alineación tarea ↔ corpus

AudioSet-EV no se mezcla con sireNNet en un único pool. El `eval` oficial es el test interno; LSSiren y UrbanSound8K son otro dominio.


In [17]:
alignment = pd.DataFrame([
    {
        "task": "deteccion_binaria",
        "train": "AudioSet-EV (unbalanced pos + negatives balanced_train)",
        "val": "holdout por yt_id desde train",
        "test_internal": "AudioSet-EV eval",
        "test_other_domain": "LSSiren (fase posterior)",
    },
    {
        "task": "tipo_multiclase",
        "train": "AudioSet-EV unbalanced positivos (puros o multi-etiqueta)",
        "val": "holdout por yt_id desde train",
        "test_internal": "AudioSet-EV eval positivos",
        "test_other_domain": "sireNNet 4 clases (test limpio opcional)",
    },
    {
        "task": "distractores_nombrados",
        "train": "no (AudioSet negativos son YouTube, no jackhammer etiquetado)",
        "val": "—",
        "test_internal": "—",
        "test_other_domain": "UrbanSound8K folds + fsID (fase posterior)",
    },
])
alignment.to_csv(TAB / "task_alignment.csv", index=False)
alignment


,task,train,val,test_internal,test_other_domain
0,deteccion_binaria,AudioSet-EV (unbalanced pos + negatives balanc...,holdout por yt_id desde train,AudioSet-EV eval,LSSiren (fase posterior)
1,tipo_multiclase,AudioSet-EV unbalanced positivos (puros o mult...,holdout por yt_id desde train,AudioSet-EV eval positivos,sireNNet 4 clases (test limpio opcional)
2,distractores_nombrados,"no (AudioSet negativos son YouTube, no jackham...",—,—,UrbanSound8K folds + fsID (fase posterior)


## Inventario de archivos montados

Si un corpus no está, se registra `missing` y se sigue.


In [18]:
paths = corpus_paths()
frames = []
status_rows = []

scanners = {
    "audioset_ev": (paths.audioset_ev, scan_audioset_ev),
    "sirennet": (paths.sirennet, scan_sirennet),
    "lssiren": (paths.lssiren, scan_lssiren),
    "urbansound8k": (paths.urbansound8k, scan_urbansound8k),
}

for name, (root, scanner) in scanners.items():
    if root is None:
        status_rows.append({"corpus": name, "status": "missing", "root": None, "n_rows": 0})
        continue
    df = scanner(root)
    frames.append(df)
    status_rows.append({"corpus": name, "status": "ok", "root": str(root), "n_rows": int(len(df))})

status = pd.DataFrame(status_rows)
status.to_csv(TAB / "mount_status.csv", index=False)
inventory = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=["corpus", "path", "relpath", "label", "task"])
if not inventory.empty:
    inventory.to_csv(TAB / "file_inventory.csv", index=False)
status


,corpus,status,root,n_rows
0,audioset_ev,ok,/home/jeancdevx/dev/doppler/doppler-ml/data/ra...,28240
1,sirennet,ok,/home/jeancdevx/dev/doppler/doppler-ml/data/ra...,1675
2,lssiren,ok,/home/jeancdevx/dev/doppler/doppler-ml/data/ra...,1834
3,urbansound8k,ok,/home/jeancdevx/dev/doppler/doppler-ml/data/ra...,8732


In [19]:
if inventory.empty:
    counts = pd.DataFrame(columns=["corpus", "label", "n"])
else:
    counts = inventory.groupby(["corpus", "label"]).size().reset_index(name="n")
counts.to_csv(TAB / "label_counts.csv", index=False)
counts


,corpus,label,n
0,audioset_ev,ambulance,934
1,audioset_ev,firetruck,2562
2,audioset_ev,multi,829
3,audioset_ev,police,2427
4,audioset_ev,siren_untyped,572
5,audioset_ev,urban_negative,20916
6,lssiren,road_noise,902
7,lssiren,siren,932
8,sirennet,ambulance,400
9,sirennet,firetruck,400


## Resultados

- AudioSet-EV es el corpus de train. sireNNet queda como piloto documentado, no como fuente de `fit`.
- `file_inventory.csv` alimenta 02–04.
- Siguiente: exploración acústica sobre AudioSet-EV.


In [20]:
result = {
    "seed": SEED,
    "n_catalog_rows": int(len(catalog)),
    "mounted": status.set_index("corpus")["status"].to_dict(),
    "n_inventory_rows": int(len(inventory)),
    "tables": [p.name for p in sorted(TAB.glob("*.csv"))],
}
result


{'seed': 7,
 'n_catalog_rows': 4,
 'mounted': {'audioset_ev': 'ok',
  'sirennet': 'ok',
  'lssiren': 'ok',
  'urbansound8k': 'ok'},
 'n_inventory_rows': 40481,
 'tables': ['corpus_catalog.csv',
  'file_inventory.csv',
  'label_counts.csv',
  'mount_status.csv',
  'task_alignment.csv']}